# Visualización de outliers y distribuciones: Problemas de Popularidad y Precios
El objetivo de este notebook es visualizar los outliers y distribuciones de todas las variables que vamos a utilizar.

---
## Importación de librerías

In [ ]:
# Módulo
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [ ]:
# Para que se puedan utilizar funciones desde el notebook
from src.utils.files import read_file
from src.utils.config import popularity, load_env_file

load_env_file()

## Carga de datos
Configuración del Minio, poner en True para usar la información de Minio

In [ ]:
use_minio = True # Solo cambiar este parámetro
minio = {"minio_write": False, "minio_read": use_minio}

In [ ]:
df = read_file(popularity, minio)
df.head(2)

---
## Análisis

Tenemos muchas columnas con diferentes tipos de datos. Las numéricas y temporales serán fáciles de visualizar, mientras que las categóricas y textuales no tanto. Vamos ir una por una analizando cada variable.

### id 

No merece la pena visualizar el id al ser un número asignado con el único propósito de identificar los juegos, por esto no se considera ni que tenga una distribución ni valores atípicos.

### name

Los nombres no tienen una distribución como tal, lo máximo que podemos hacer es visualizar la longitud de los mismos. Se observa que la media está en torno a los 13 caracteres, teniendo algunos valores atípicos con más de 100 caracteres.

In [ ]:
df_names = df.copy()
df_names["len_name"] = df_names["name"].apply(len)
df_names = df_names[df_names["len_name"] < 150]
fig = px.histogram(df_names, x = df_names["len_name"], hover_name= df_names["name"], color_discrete_sequence  = ["lightseagreen"], title="<b>Distribución de longitudes de nombres</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()

### short_description
No merece la pena visualizar la descripción de los juegos ya que no hay ninguna manera directa de ver distribuciones o valores atípicos.

### price_overview y price_range
La distribución de precios ya se muestra en el notebook dedicado a ello, además de experimentar con diferentes condiciones. Pero repasamos la distribución total de la variable. Destaca el rango de precios de 0.01€ a 4.99€ con más del 40% de los juegos. Además de que menos del 25% de los juegos cuesta más de 10€.

In [ ]:
fig = go.Figure()
orden = [
    "Free", "[0.01,4.99]",
    "[5.00,9.99]", "[10.00,14.99]",
    "[15.00,19.99]", "[20.00,29.99]",
    "[30.00,39.99]", ">40"]

values_df = df["price_range"].value_counts()
values_df = values_df.reindex(orden)

fig.add_trace(go.Pie(values= values_df.values, labels= values_df.index, sort = False))
fig.update_layout(
    width = 800,
    title = {
        "text": "Proporción de los rangos de precio",
        "x": 0.5, "y":0.9, "font":dict(family="Verdana", size = 20, weight="bold")
    })

fig.show()

### num_languages
Esta es una variable que almacena el número de idiomas que soporta un juego.

In [ ]:
fig = px.histogram(df, x = df["num_languages"], hover_name= df["name"], color_discrete_sequence  = ["lightseagreen"], title="<b>Distribución de número de lenguages soportados</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()

La mayoría de los juegos sólo se encuentran disponibles en 1 o 2 idiomas como máximo mientras que tan sólo unos pocos son traducidos a una gran variedad de idiomas (normalmente estos son juegos exitosos que tienen los recursos para permitirse las traducciones). Por último destaca un grupo de juegos que indican tener todas las traducciones permitidas en Steam, probablemente esto no sea cierto y sea una estrategia para que dichos juegos lleguen a un mayor público.

### developers y publishers
Vamos a intentar hacer lo mismo que con los idomas, pero hay muchos más desarrolladores y publishers que idomas, por lo que visualizaremos solo los que tengan más de cierto número de juegos.

In [ ]:
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=("Distribución de Developers", "Distribución de Publishers"))

fig.add_trace(
    go.Histogram(
        x= df["total_games_by_developer"],
        marker_color= "lightseagreen",
        xbins = dict(size = 3)
), row = 1, col = 1)

fig.add_trace(
    go.Histogram(
        x=df["total_games_by_publisher"],
        marker_color= "steelblue",
        xbins = dict(size = 3)
),row = 1, col = 2)

fig.update_layout(height=800, width=1200, showlegend=False,)
fig.update_xaxes(title_text="Cantidad de juegos lanzados", range = [0, min(df["total_games_by_developer"].max(), df["total_games_by_publisher"].max())])
fig.update_yaxes(title_text="Número total", type = "log")
fig.show()

Las distribución tanto del número de juegos lanzados por publishers como developers es muy parecida, ambos muestran una distribución de cola larga con la excepción de que en este caso sí que existe un grupo de outliers relativamente grande conformado por developers/publishers que tienen una gran cantidad de juegos lanzados. Excluyendo a las grandes empresas de la industria de videojuegos, hemos comprobado que existen algunas compañías que lanzan cientos de juegos del mismo género y que son prácticamente idénticos entre sí.

### release_year
Ver la distribución de esta variable es sencillo

In [ ]:
df_release = df.copy()

df_release["release_year"] = df_release["release_year"].astype(int)

all_years = sorted(df["release_year"].unique())

fig = px.histogram(
    df_release,
    x="release_year",
    title="<b>Distribución de años de lanzamiento</b>",
    labels={'release_year': 'Año', 'count': 'Número de juegos'},
    color_discrete_sequence=["lightcoral"]
)

fig.update_layout(
    title={'x': 0.5, 'y': 0.9, 'xanchor': 'center', 'yanchor': 'top',
           'font': dict(family="Verdana", size=20)},
    bargap=0.1,
    plot_bgcolor="WhiteSmoke",
    xaxis=dict(
        tickmode="array",      # usamos un array de ticks
        tickvals=all_years,    # valores que queremos mostrar
        ticktext=[str(y) for y in all_years]  # texto de cada tick
    )
)

fig.show()

### recomendaciones_totales
La distribución es una cola larga, teniendo casi todos los juegos muy pocas reseñas.

In [ ]:
limit = df["recomendaciones_totales"].quantile(0.98)
bins = np.linspace(0, limit, 25)
hist, _ = np.histogram(df["recomendaciones_totales"], bins=bins)
labels = [f"{int(bins[i])}-{int(bins[i+1])}" for i in range(len(bins)-1)]
hist_log = np.log1p(hist)


fig = go.Figure()

fig.add_trace(go.Bar(
    y=labels,
    x=hist_log,
    orientation="h",
    marker_color="seagreen",
    hovertemplate="Rango: %{y}<br>Juegos: %{customdata}",
    customdata=hist
))


tick_vals_raw = [0, 10, 100, 1000, 10000]
tick_vals_log = np.log1p(tick_vals_raw)

fig.update_layout(
    title={"text": "<b>Distribución de Recomendaciones Totales (Escala Log)</b>", "x": 0.5},
    xaxis=dict(
        tickvals=tick_vals_log,
        ticktext=[str(x) for x in tick_vals_raw],
        title="Número de juegos (Escala Log)"
    ),
    yaxis=dict(title="Rango de recomendaciones"),
    height=700
)

fig.show()

In [ ]:
print(f"Sólo el {round((df[df["recomendaciones_totales"] > 50].shape[0] / df.shape[0]) * 100,2)}% de los juegos supera las 50 reseñas")

### brillo
El brillo medio es un valor que va entre 0 y 255 (valores de negro y blanco), la media está entre 50 y 100 de brillo

In [ ]:
fig = px.histogram(df, x = df["brillo"], hover_name= df["name"], color_discrete_sequence  = ["crimson"], title="<b>Distribución del brillo de las cápsulas</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()

### yt_score
El yt_score es una ponderación normalizada, por lo que está entre 0 y 1. Destaca que casi el 25% de los juegos tiene un score de 0, es decir, no tienen ningún video.

In [ ]:
fig = px.histogram(df, x = df["yt_score"], hover_name= df["name"], color_discrete_sequence  = ["crimson"], title="<b>Distribución yt_score</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()